# Notebook 01 — Data Cleaning Walkthrough
## Social Media Engagement Analytics Dashboard

This notebook walks through every step of the data-cleaning pipeline
in an interactive, visual way.

The same logic is implemented as reusable functions in `src/clean_data.py`.
This notebook is for **learning and exploring** the cleaning process.

### How to run
1. Make sure you have run `python main.py generate-sample` first
2. Run each cell in order (Shift + Enter)

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the project root to the Python path
sys.path.insert(0, str(Path('..').resolve()))

from src.config import settings
from src.utils import parse_iso8601_duration

print('✓ Imports complete')
print(f'Data path: {settings["RAW_DATA_PATH"]}')

## Step 1 — Load Raw Data
Let's look at what the raw dataset looks like before any cleaning.

In [ ]:
raw_path = settings['RAW_DATA_PATH']

if not raw_path.exists():
    print('Raw data not found. Run: python main.py generate-sample')
else:
    df = pd.read_csv(raw_path)
    print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
    df.head(3)

In [ ]:
# Check data types
print('Column data types:')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(f'Columns with missing values: {len(missing_df)}')
missing_df

In [ ]:
# Check for duplicates
print(f'Exact duplicate rows:  {df.duplicated().sum()}')
if 'video_id' in df.columns:
    print(f'Duplicate video_ids:   {df["video_id"].duplicated().sum()}')

## Step 2 — Apply Cleaning Pipeline
Now we run the full cleaning pipeline from `src/clean_data.py`.

In [ ]:
from src.clean_data import clean_data
from src.feature_engineering import engineer_features

# Run cleaning
df_clean = clean_data()
print(f'\nCleaned shape: {df_clean.shape}')

In [ ]:
# Run feature engineering
df_final = engineer_features(df_clean)
print(f'Final shape: {df_final.shape}')
print(f'New columns added: {df_final.shape[1] - df_clean.shape[1]}')

## Step 3 — Before vs After Comparison

In [ ]:
print('=== DATA QUALITY REPORT ===')
print(f'Initial rows       : {len(df)}')
print(f'After cleaning     : {len(df_clean)}')
print(f'After engineering  : {len(df_final)}')
print(f'Total columns      : {df_final.shape[1]}')
print(f'\nMissing values (after cleaning): {df_final.isnull().sum().sum()}')

# Engagement stats
print(f'\nEngagement Rate Stats:')
print(df_final['engagement_rate'].describe().round(4))

In [ ]:
# Visualise: view count distribution before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['view_count'].dropna(), bins=30, color='#2E86AB', edgecolor='white')
axes[0].set_title('View Count Distribution — Raw Data', fontsize=13, fontweight='bold')
axes[0].set_xlabel('View Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_final['view_count'], bins=30, color='#4CAF50', edgecolor='white')
axes[1].set_title('View Count Distribution — Cleaned Data', fontsize=13, fontweight='bold')
axes[1].set_xlabel('View Count')
axes[1].set_ylabel('Frequency')

plt.suptitle('View Count: Before vs After Cleaning', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/view_count_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Most YouTube data follows a heavily right-skewed distribution —')
print('a few viral videos have millions of views while most have far fewer.')

In [ ]:
# Save final processed data
out_path = settings['PROCESSED_DATA_PATH']
out_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(out_path, index=False)
print(f'✓ Processed data saved: {out_path}')
print('\nFinal columns:')
for i, col in enumerate(df_final.columns):
    print(f'  {i+1:2d}. {col}')